In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

In [12]:
from Utils.Reshape import reshape
import torch

t = reshape(split = [2,2,2], map_type= 1, device = 'cuda')

x = torch.rand((10,256,10,10)).to('cuda')

o = t(x)
o.shape

torch.Size([10, 2, 2, 2, 128, 5, 5])

In [9]:
class ChunkReshape:
    def __init__(self, chunk_divide_sizes, map_type=1, device='cpu'):
        self.chunk_divide_sizes = chunk_divide_sizes
        self.map_type = map_type
        self.device = device
    
    def split_into_chunks(self, tensor):
        batch_size, C, H, W = tensor.shape
        chunks = []
        
        if self.map_type == 1:
            C_indices, H_indices, W_indices = [
                [sum(dim // self.chunk_divide_sizes[i] for _ in range(j)) for j in range(self.chunk_divide_sizes[i] + 1)]
                for i, dim in enumerate([C, H, W])
            ]
            
            for b in range(batch_size):
                for i in range(self.chunk_divide_sizes[0]):
                    for j in range(self.chunk_divide_sizes[1]):
                        for k in range(self.chunk_divide_sizes[2]):
                            chunk = tensor[b,
                                           C_indices[i]:C_indices[i+1], 
                                           H_indices[j]:H_indices[j+1], 
                                           W_indices[k]:W_indices[k+1]].unsqueeze(0)
                            chunks.append(chunk)

        elif self.map_type == 2:
            for b in range(batch_size):
                for i in range(self.chunk_divide_sizes[0]):
                    for j in range(self.chunk_divide_sizes[1]):
                        for k in range(self.chunk_divide_sizes[2]):
                            C_stride_indices = torch.arange(i, C, self.chunk_divide_sizes[0]).to(self.device)
                            H_stride_indices = torch.arange(j, H, self.chunk_divide_sizes[1]).to(self.device)
                            W_stride_indices = torch.arange(k, W, self.chunk_divide_sizes[2]).to(self.device)

                            chunk = tensor[b].index_select(0, C_stride_indices)
                            chunk = chunk.index_select(1, H_stride_indices)
                            chunk = chunk.index_select(2, W_stride_indices).unsqueeze(0)
                            chunks.append(chunk)
        
        return chunks

    def stack_chunks_to_form_tensor(self, chunks):
        batch_size = len(chunks) // (self.chunk_divide_sizes[0] * self.chunk_divide_sizes[1] * self.chunk_divide_sizes[2])
        result = torch.cat(chunks).view(
            batch_size, self.chunk_divide_sizes[0], self.chunk_divide_sizes[1], self.chunk_divide_sizes[2], 
            *chunks[0].shape[1:])
        return result

In [10]:
chunk_manager = ChunkReshape([2, 2, 2], map_type=2, device='cuda')
chunks = chunk_manager.split_into_chunks(x)
final_tensor_map_type1 = chunk_manager.stack_chunks_to_form_tensor(chunks)

print('Are the results still the same as before ? ', torch.equal(o, final_tensor_map_type1))


Are the results still the same as before ?  True


In [11]:
o.device

device(type='cuda', index=0)

In [13]:
device = 'cuda'
for i in range(2):
    for j in range(2):
        for k in range(2):
              chunk = torch.index_select(
              torch.index_select(
              torch.index_select(
              x
              , 1, torch.arange(0,128,1).to(device) if i==0 else torch.arange(128,256,1).to(device)).to(device)
              , 2, torch.arange(0,5,1).to(device) if j==0 else torch.arange(5,10,1).to(device)).to(device)
              , 3, torch.arange(0,5,1).to(device) if k==0 else torch.arange(5,10,1).to(device)).to(device)

              if k==0:
                 memory = chunk

              if k==1 and j==0:
                 memory2 = torch.stack((memory, chunk), dim = 1).to(device)
              if k==1 and j==1 and i==0:
                 memory3 = torch.stack((memory2, torch.stack((memory, chunk), dim = 1)), dim = 1).to(device)
              if k==1 and j==1 and i==1:
                 output = torch.stack((memory3, torch.stack((memory2, torch.stack((memory, chunk), dim = 1)), dim = 1)), dim = 1).to(device)


print('Are the results still the same as before ? ', torch.equal(output, o))


Are the results still the same as before ?  True


In [35]:
o

tensor([[[[[[[2.5079e-01, 9.0247e-01, 6.5132e-01, 8.6035e-01, 6.1527e-01],
             [1.4335e-01, 4.3423e-01, 3.1444e-03, 8.1364e-01, 4.2882e-01],
             [6.2964e-01, 3.7483e-01, 5.2597e-01, 2.3361e-01, 9.6528e-01],
             [2.9083e-01, 3.5029e-01, 4.9116e-01, 2.9622e-02, 6.9685e-01],
             [5.8027e-02, 9.9667e-01, 2.6757e-01, 6.8624e-01, 7.9922e-01]],

            [[7.3955e-01, 7.4002e-02, 1.2251e-01, 4.0702e-01, 8.7002e-01],
             [7.4405e-01, 2.6099e-01, 3.3500e-01, 7.4135e-01, 8.9981e-02],
             [5.1031e-01, 6.8247e-01, 3.4336e-01, 5.4080e-01, 4.9460e-01],
             [8.4324e-01, 8.9302e-01, 2.7906e-01, 8.6177e-01, 6.1545e-01],
             [4.9593e-02, 7.2567e-01, 4.9479e-01, 4.7209e-01, 7.6314e-01]],

            [[9.3775e-01, 4.0769e-01, 4.2230e-01, 5.6332e-01, 1.8428e-01],
             [3.3483e-01, 1.5279e-01, 7.2756e-01, 5.6374e-01, 1.4506e-01],
             [1.4912e-01, 5.5778e-01, 2.2207e-02, 7.9543e-01, 7.0959e-01],
             [2.1154e

In [34]:
output

tensor([[[[[[[2.5079e-01, 9.0247e-01, 6.5132e-01, 8.6035e-01, 6.1527e-01],
             [1.4335e-01, 4.3423e-01, 3.1444e-03, 8.1364e-01, 4.2882e-01],
             [6.2964e-01, 3.7483e-01, 5.2597e-01, 2.3361e-01, 9.6528e-01],
             [2.9083e-01, 3.5029e-01, 4.9116e-01, 2.9622e-02, 6.9685e-01],
             [5.8027e-02, 9.9667e-01, 2.6757e-01, 6.8624e-01, 7.9922e-01]],

            [[7.3955e-01, 7.4002e-02, 1.2251e-01, 4.0702e-01, 8.7002e-01],
             [7.4405e-01, 2.6099e-01, 3.3500e-01, 7.4135e-01, 8.9981e-02],
             [5.1031e-01, 6.8247e-01, 3.4336e-01, 5.4080e-01, 4.9460e-01],
             [8.4324e-01, 8.9302e-01, 2.7906e-01, 8.6177e-01, 6.1545e-01],
             [4.9593e-02, 7.2567e-01, 4.9479e-01, 4.7209e-01, 7.6314e-01]],

            [[9.3775e-01, 4.0769e-01, 4.2230e-01, 5.6332e-01, 1.8428e-01],
             [3.3483e-01, 1.5279e-01, 7.2756e-01, 5.6374e-01, 1.4506e-01],
             [1.4912e-01, 5.5778e-01, 2.2207e-02, 7.9543e-01, 7.0959e-01],
             [2.1154e

In [2]:
from Utils.Accuracy_measures import topk_accuracy
from Utils.TinyImageNet_loader import get_tinyimagenet_dataloaders
from Utils.Num_parameter import count_parameters
from Models.Resnet50 import Resnet50

import torchvision.transforms as transforms
from torch import nn
from torch import optim

import time
import torch
import os

In [3]:
import tltorch
new_classifier = nn.Sequential(
        tltorch.TRL(input_shape=(2048,6,6), output_shape=(10), factorization='Tucker', rank=(100,3,3,10))
    ) 

model = Resnet50(pretrained=False,
                          weights_path='../weights/resnet50_weights.pth',
                          tensorized=True,
                          input_shape=(192,192),
                          num_classes=10,
                          avg_pool=False,
                          new_classifier=new_classifier).to('cpu')

t = torch.rand((5,3,192,192))
o = model(t)
o.shape

torch.Size([5, 10])

In [1]:
raise(ValueError('Wrong shapes'))

ValueError: Wrong shapes

In [5]:
layer = 0
for child in model.children():
    layer+=1
    
    if layer < 3:
        print(f'{layer}  {child} \n\n')
        # for param in child.parameters():
        #     param.requires_grad = False

1  Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=Fals

In [18]:
from Models.CNN import out_shape_cnn
out_shape_cnn(in_shape=(192,192), batch_size= 5)

torch.Size([5, 128, 6, 6])

In [77]:
import tltorch
# 512 6 6 ->   
# tcl = tltorch.TCL(input_shape=(512,6,6), rank=(2,1,1,10,2,2)) 
trl = tltorch.TRL(input_shape=(2,1,1,40,2,2), output_shape=(200), factorization='Tucker', rank=(1,1,1,10,1,1,200))

# print(f'tcl parameters : {count_parameters(tcl)}')
print(f'trl parameters : {count_parameters(trl)}')
# print(f'tcl+trl parameters : {count_parameters(tcl) + count_parameters(trl)}')
# print(f'saving :  {(1-((count_parameters(tcl) + count_parameters(trl))/ classifier_parameters))*100}')

trl parameters : 42408
